In [ ]:
%%capture
!python /kaggle/usr/lib/script1/script1.py

In [ ]:
%%capture
!python /kaggle/usr/lib/0_585/0_585.py

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import gc
import sys
import joblib
import subprocess
import numpy as np
import pandas as pd
import polars as pl
from glob import glob
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
df_train,y,df_test=joblib.load('/kaggle/working/data.pkl')

In [ ]:
fitted_models_lgb=[]
model = lgb.LGBMClassifier()
model.fit(df_train,y)
fitted_models_lgb.append(model)  

In [ ]:
class VotingModel(BaseEstimator, RegressorMixin):
    def __init__(self, estimators):
        super().__init__()
        self.estimators = estimators
        
    def fit(self, X, y=None):
        return self
    
    def predict(self, X):
        y_preds = [estimator.predict(X) for estimator in self.estimators]
        return np.mean(y_preds, axis=0)
    
    def predict_proba(self, X):
        y_preds = [estimator.predict_proba(X) for estimator in self.estimators]
        
        return np.mean(y_preds, axis=0)

model = VotingModel(fitted_models_lgb)

In [ ]:
df_test = df_test.drop(columns=["WEEK_NUM",'target'])
df_test = df_test.set_index("case_id")

y_pred = pd.Series(model.predict_proba(df_test)[:,1], index=df_test.index)
condition=y_pred<0.98
df_subm = pd.read_csv("/kaggle/working/sub.csv")
df_subm = df_subm.set_index("case_id")

SHIFT = 0.072

df_subm.loc[condition, 'score'] = (df_subm.loc[condition, 'score'] - SHIFT).clip(0)
df_subm.to_csv("submission.csv")
df_subm
!rm -rf data.pkl